In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn import over_sampling
from imblearn.over_sampling import SMOTE
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score,recall_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
df=pd.read_csv('df_75.csv')

In [3]:
X = df.drop('TARGET',axis=1)
y = df['TARGET']

from imblearn import over_sampling
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=10,sampling_strategy=0.25)
X,y = smote.fit_resample(X,y)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=10000, test_size = 0.2)
print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
X_train.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

X_train_num = X_train.select_dtypes(include=['int64','float64'])
X_train_cat = X_train.select_dtypes(include='uint8')
X_test_num = X_test.select_dtypes(include=['int64','float64'])
X_test_cat = X_test.select_dtypes(include='uint8')

ss = StandardScaler()
X_train_num = pd.DataFrame(ss.fit_transform(X_train_num), columns=X_train_num.columns)
X_test_num = pd.DataFrame(ss.fit_transform(X_test_num), columns=X_test_num.columns)

X_train = pd.concat([X_train_num,X_train_cat], axis=1)
X_test = pd.concat([X_test_num,X_test_cat], axis=1)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)

print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train: (140176, 108)
y_train: (140176,)
X_train: (140176, 108)
y_train: (140176,)
X_test: (35045, 108)
y_test: (35045,)


In [35]:
lr = LogisticRegression()

lr_model = lr.fit(X_train,y_train)

In [36]:
f1_score(y_train,lr_model.predict(X_train))

0.5698008534206966

In [37]:
f1_score(y_test,lr_model.predict(X_test))

0.5696446938665032

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

In [5]:
Ran_clf = RandomForestClassifier()
Ran_clf=Ran_clf.fit(X_train, y_train)

In [6]:
y_pred_ran = Ran_clf.predict(X_test)

In [7]:
print(classification_report(y_test,y_pred_ran))

              precision    recall  f1-score   support

           0       0.90      1.00      0.95     28058
           1       0.98      0.55      0.71      6987

    accuracy                           0.91     35045
   macro avg       0.94      0.77      0.83     35045
weighted avg       0.91      0.91      0.90     35045



In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

In [9]:
clf = DecisionTreeClassifier()

In [10]:
clf = clf.fit(X_train,y_train)

In [11]:
clf.score(X_train,y_train)

1.0

In [12]:
y_pred_dt = clf.predict(X_test)

In [13]:
y_pred_dt = [ 0 if x < 0.5 else 1 for x in y_pred_dt]
y_pred_dt[0:5]

[0, 1, 1, 0, 0]

In [14]:
print(classification_report(y_test,y_pred_dt))

              precision    recall  f1-score   support

           0       0.89      0.75      0.82     28058
           1       0.39      0.64      0.49      6987

    accuracy                           0.73     35045
   macro avg       0.64      0.70      0.65     35045
weighted avg       0.79      0.73      0.75     35045



In [15]:
from sklearn.ensemble import GradientBoostingClassifier
gbc = GradientBoostingClassifier()
gbc_clf=gbc.fit(X_train,y_train)

In [16]:
y_pred_gbc = gbc_clf.predict(X_test)
y_pred_gbc = [ 0 if x < 0.5 else 1 for x in y_pred_gbc]
print(classification_report(y_test,y_pred_gbc))

              precision    recall  f1-score   support

           0       0.90      0.99      0.94     28058
           1       0.90      0.54      0.68      6987

    accuracy                           0.90     35045
   macro avg       0.90      0.76      0.81     35045
weighted avg       0.90      0.90      0.89     35045



In [18]:
from sklearn.ensemble import AdaBoostClassifier
ada = AdaBoostClassifier()
ada_clf=ada.fit(X_train,y_train)

In [19]:
y_pred_ada=ada_clf.predict(X_test)
y_pred_ada=[ 0 if x < 0.5 else 1 for x in y_pred_ada]
print(classification_report(y_test,y_pred_ada))

              precision    recall  f1-score   support

           0       0.88      0.99      0.93     28058
           1       0.91      0.48      0.63      6987

    accuracy                           0.89     35045
   macro avg       0.90      0.73      0.78     35045
weighted avg       0.89      0.89      0.87     35045



In [20]:
from xgboost import XGBClassifier
classifier = XGBClassifier()
XGB_clf=classifier.fit(X_train, y_train)

C:\Users\mahes\anaconda3\lib\site-packages\xgboost\sklearn.py:1146: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[23:37:41] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.4.0/src/learner.cc:1095: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


In [21]:
y_pred_XGB = XGB_clf.predict(X_test)
y_pred_XGB=[ 0 if x < 0.5 else 1 for x in y_pred_XGB]
print(classification_report(y_test,y_pred_XGB))

              precision    recall  f1-score   support

           0       0.95      0.11      0.20     28058
           1       0.22      0.97      0.35      6987

    accuracy                           0.29     35045
   macro avg       0.58      0.54      0.28     35045
weighted avg       0.80      0.29      0.23     35045



In [24]:
from sklearn.ensemble import StackingClassifier

base=[('Decision Tree',DecisionTreeClassifier(min_samples_leaf=3,max_depth=25,random_state=20)),
      ('Random Forest',RandomForestClassifier(random_state=20))]

stack=StackingClassifier(estimators=base,final_estimator=GradientBoostingClassifier(random_state=20))
stack_model=stack.fit(X_train,y_train)

In [25]:
y_pred_stack = stack_model.predict(X_test)
y_pred_stack=[ 0 if x < 0.5 else 1 for x in y_pred_stack]
print(classification_report(y_test,y_pred_stack))

              precision    recall  f1-score   support

           0       0.91      0.98      0.94     28058
           1       0.89      0.61      0.72      6987

    accuracy                           0.91     35045
   macro avg       0.90      0.79      0.83     35045
weighted avg       0.90      0.91      0.90     35045



##RFE(Feature selection)

In [28]:
dt=DecisionTreeClassifier()

In [29]:
from sklearn.feature_selection import RFE
rfe = RFE(dt , n_features_to_select = 30)
rfe.fit(X_train , y_train)

RFE(estimator=DecisionTreeClassifier(), n_features_to_select=30)

In [30]:
rank = pd.DataFrame(rfe.ranking_ , index = X_train.columns , columns = ['Ranking'])

In [31]:
selected_columns = rank[rank['Ranking']==1].index
selected_columns

Index(['REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED',
       'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE',
       'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
       'AMT_REQ_CREDIT_BUREAU_YEAR', 'LOG_AMT_INCOME_TOTAL', 'FLAG_DOC_SCORE',
       'EXT_SOURCE_AVG', 'LOG_AMT_CREDIT', 'LOG_AMT_ANNUITY',
       'LOG_AMT_GOODS_PRICE', 'DAYS_LAST_PHONE_CHANGE_TR', 'FLAG_OWN_CAR_Y',
       'NAME_EDUCATION_TYPE_Higher education',
       'NAME_EDUCATION_TYPE_Incomplete higher',
       'NAME_EDUCATION_TYPE_Lower secondary',
       'NAME_EDUCATION_TYPE_Secondary / secondary special',
       'NAME_FAMILY_STATUS_Married', 'NAME_FAMILY_STATUS_Separated',
       'NAME_FAMILY_STATUS_Single / not married', 'NAME_FAMILY_STATUS_Widow',
       'REGION_RATING_CLIENT_2', 'REGION_RATING_CLIENT_3',
       'WEEKDAY_APPR_PROCESS_START_THURSDAY',
       'WEEKDAY_APPR_PROCESS_START_TUESDAY',
       'WEEKDAY_APPR_PROCESS_START_WEDNESDAY'],
      dtype='object')

In [32]:
df1=pd.DataFrame()
for i in selected_columns:
    df1[i]=df[i]
df1.head()

,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,LOG_AMT_INCOME_TOTAL,...,NAME_EDUCATION_TYPE_Secondary / secondary special,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Widow,REGION_RATING_CLIENT_2,REGION_RATING_CLIENT_3,WEEKDAY_APPR_PROCESS_START_THURSDAY,WEEKDAY_APPR_PROCESS_START_TUESDAY,WEEKDAY_APPR_PROCESS_START_WEDNESDAY
0,0.100160,-19046,-225,65.268675,-2531,26.0,0.0,0.0,0.0,11.119883,...,1,0,0,1,0,1,0,0,0,0
1,0.089549,-19005,-3039,99.161484,-2437,-1.0,0.8,0.2,1.9,11.813030,...,1,0,0,0,0,1,0,0,0,1
2,0.169302,-19932,-3038,65.658206,-3458,-1.0,0.0,0.0,0.0,11.707670,...,1,0,0,1,0,1,0,1,0,0
3,0.189188,-16941,-1588,70.498227,-477,-1.0,0.0,1.0,1.0,11.502875,...,1,1,0,0,0,1,0,0,0,1
4,0.189188,-13778,-3130,34.828150,-619,17.0,1.0,1.0,2.0,12.049419,...,0,1,0,0,0,1,0,0,0,0


In [33]:
X = df1
y = df['TARGET']

from imblearn import over_sampling
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=10,sampling_strategy=0.25)
X,y = smote.fit_resample(X,y)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=10000, test_size = 0.2)
print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
X_train.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

X_train_num = X_train.select_dtypes(include=['int64','float64'])
X_train_cat = X_train.select_dtypes(include='uint8')
X_test_num = X_test.select_dtypes(include=['int64','float64'])
X_test_cat = X_test.select_dtypes(include='uint8')

ss = StandardScaler()
X_train_num = pd.DataFrame(ss.fit_transform(X_train_num), columns=X_train_num.columns)
X_test_num = pd.DataFrame(ss.fit_transform(X_test_num), columns=X_test_num.columns)

X_train = pd.concat([X_train_num,X_train_cat], axis=1)
X_test = pd.concat([X_test_num,X_test_cat], axis=1)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)

print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train: (140176, 30)
y_train: (140176,)
X_train: (140176, 30)
y_train: (140176,)
X_test: (35045, 30)
y_test: (35045,)


In [30]:
lr = LogisticRegression()

lr_model = lr.fit(X_train,y_train)

In [31]:
f1_score(y_train,lr_model.predict(X_train))

0.5680920425342572

In [32]:
f1_score(y_test,lr_model.predict(X_test))

0.568019439306568

In [34]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

In [35]:
Ran_clf = RandomForestClassifier()
Ran_clf=Ran_clf.fit(X_train, y_train)

In [36]:
y_pred_ran = Ran_clf.predict(X_test)

In [37]:
print(classification_report(y_test,y_pred_ran))

              precision    recall  f1-score   support

           0       0.90      0.98      0.93     28058
           1       0.85      0.55      0.67      6987

    accuracy                           0.89     35045
   macro avg       0.87      0.76      0.80     35045
weighted avg       0.89      0.89      0.88     35045



In [38]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

In [39]:
clf = DecisionTreeClassifier()

In [40]:
clf = clf.fit(X_train,y_train)

In [41]:
clf.score(X_train,y_train)

1.0

In [42]:
y_pred_dt = clf.predict(X_test)

In [43]:
y_pred_dt = [ 0 if x < 0.5 else 1 for x in y_pred_dt]
y_pred_dt[0:5]

[0, 1, 1, 0, 0]

In [44]:
print(classification_report(y_test,y_pred_dt))

              precision    recall  f1-score   support

           0       0.89      0.84      0.86     28058
           1       0.47      0.58      0.52      6987

    accuracy                           0.78     35045
   macro avg       0.68      0.71      0.69     35045
weighted avg       0.80      0.78      0.79     35045



In [45]:
from sklearn.ensemble import GradientBoostingClassifier
gbc = GradientBoostingClassifier()
gbc_clf=gbc.fit(X_train,y_train)

In [46]:
y_pred_gbc = gbc_clf.predict(X_test)
y_pred_gbc = [ 0 if x < 0.5 else 1 for x in y_pred_gbc]
print(classification_report(y_test,y_pred_gbc))

              precision    recall  f1-score   support

           0       0.89      0.94      0.91     28058
           1       0.70      0.52      0.59      6987

    accuracy                           0.86     35045
   macro avg       0.79      0.73      0.75     35045
weighted avg       0.85      0.86      0.85     35045



In [47]:
from sklearn.ensemble import AdaBoostClassifier
ada = AdaBoostClassifier()
ada_clf=ada.fit(X_train,y_train)

In [48]:
y_pred_ada=ada_clf.predict(X_test)
y_pred_ada=[ 0 if x < 0.5 else 1 for x in y_pred_ada]
print(classification_report(y_test,y_pred_ada))

              precision    recall  f1-score   support

           0       0.88      0.99      0.93     28058
           1       0.89      0.44      0.59      6987

    accuracy                           0.88     35045
   macro avg       0.88      0.72      0.76     35045
weighted avg       0.88      0.88      0.86     35045



In [49]:
from xgboost import XGBClassifier
classifier = XGBClassifier()
XGB_clf=classifier.fit(X_train, y_train)

C:\Users\mahes\anaconda3\lib\site-packages\xgboost\sklearn.py:1146: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[00:37:42] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.4.0/src/learner.cc:1095: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


In [50]:
y_pred_XGB = XGB_clf.predict(X_test)
y_pred_XGB=[ 0 if x < 0.5 else 1 for x in y_pred_XGB]
print(classification_report(y_test,y_pred_XGB))

              precision    recall  f1-score   support

           0       0.94      0.26      0.41     28058
           1       0.24      0.93      0.38      6987

    accuracy                           0.40     35045
   macro avg       0.59      0.60      0.40     35045
weighted avg       0.80      0.40      0.40     35045



In [51]:
from sklearn.ensemble import StackingClassifier

base=[('Decision Tree',DecisionTreeClassifier(min_samples_leaf=3,max_depth=25,random_state=20)),
      ('Random Forest',RandomForestClassifier(random_state=20))]

stack=StackingClassifier(estimators=base,final_estimator=GradientBoostingClassifier(random_state=20))
stack_model=stack.fit(X_train,y_train)

In [52]:
y_pred_stack = stack_model.predict(X_test)
y_pred_stack=[ 0 if x < 0.5 else 1 for x in y_pred_stack]
print(classification_report(y_test,y_pred_stack))

              precision    recall  f1-score   support

           0       0.91      0.88      0.89     28058
           1       0.57      0.67      0.62      6987

    accuracy                           0.83     35045
   macro avg       0.74      0.77      0.75     35045
weighted avg       0.85      0.83      0.84     35045

